# AIC2024 KIS — Object tagging bằng YOLO11 COCO (Kaggle)

Notebook **self-contained**: đọc bundle keyframe (`manifest.jsonl` + `keyframes/`), chạy
YOLO11 pretrained COCO trên từng keyframe, xuất **object shard theo video**.

## Object lưu ở đâu?

Artifact gốc là JSONL (dễ inspect, không cần SQLite):

```
objects_shards/
  shards/<video_id>/objects.jsonl
  shards/<video_id>/manifest.json
```

Mỗi dòng trong `objects.jsonl`:

```json
{"kf_uid": 123, "video_id": "L01_V001", "generation": 0, "shot_idx": 4, "kf_idx": 1, "ts_ms": 16000, "objects": ["car", "person"], "counts": {"car": 2, "person": 1}}
```

- `objects`: nhãn COCO duy nhất (lowercase, đã sort).
- `counts`: số instance mỗi nhãn trong keyframe.
- Không lưu bbox để file gọn; filter chỉ cần nhãn + count.

Ở local, import vào SQLite để filter nhanh:

```bash
uv run ir import-objects artifacts_objects_shards.zip
uv run ir objects                              # xem nhãn + số keyframe
uv run ir search "..." --object person --object car
```

## Chuẩn bị

1. Ở local tạo bundle: `uv run ir export-keyframes` → `artifacts_keyframes.zip`.
2. Upload zip đó thành Kaggle Dataset.
3. Notebook Settings: bật **GPU** và **Internet**.
4. Add dataset, chọn batch (bên dưới), chạy hết cell.
5. Tải `artifacts_objects_shards.zip` về máy rồi import:
   `uv run ir import-objects <zip>`.

## Chọn batch

- Mặc định: tất cả video trong manifest.
- Theo lô: `OBJECTS_VIDEO_OFFSET` + `OBJECTS_VIDEO_LIMIT` (ví dụ 0/100, 100/100, ...).
- Hoặc chỉ định: `OBJECTS_VIDEO_IDS=L01_V001,L01_V002`.
- Resume: shard đã xong trong `WORK_DIR` sẽ được bỏ qua.

## Output mỗi shard

`shards/<video_id>/objects.jsonl`, `manifest.json` (model + tham số + `by_label`).


## 1. Cấu hình

In [ ]:
import os
from pathlib import Path

print("START: cấu hình", flush=True)

MODEL_NAME = os.environ.get("OBJECTS_MODEL", "yolo11x.pt")
CONF = float(os.environ.get("OBJECTS_CONF", "0.25"))
IOU = float(os.environ.get("OBJECTS_IOU", "0.45"))
IMGSZ = int(os.environ.get("OBJECTS_IMGSZ", "640"))
MAX_DET = int(os.environ.get("OBJECTS_MAX_DET", "300"))
BATCH_SIZE = int(os.environ.get("OBJECTS_BATCH_SIZE", "16"))
NUM_WORKERS = int(os.environ.get("OBJECTS_NUM_WORKERS", "4"))
PROGRESS_EVERY = int(os.environ.get("OBJECTS_PROGRESS_EVERY", "10"))
CLASSES = [
    value.strip().lower()
    for value in os.environ.get("OBJECTS_CLASSES", "").split(",")
    if value.strip()
]

BUNDLE_DIR = os.environ.get("OBJECTS_BUNDLE_DIR") or None
WORK_DIR = Path(os.environ.get("OBJECTS_WORK_DIR", "/kaggle/working/objects-yolo11"))
SHARDS_DIR = Path(os.environ.get("OBJECTS_SHARDS_DIR", str(WORK_DIR / "shards")))
ZIP_PATH = Path(
    os.environ.get(
        "OBJECTS_ZIP_PATH",
        "/kaggle/working/artifacts_objects_shards.zip",
    )
)
RESUME = os.environ.get("OBJECTS_RESUME", "1") != "0"
REQUIRE_GPU = os.environ.get("OBJECTS_REQUIRE_GPU", "1") != "0"

VIDEO_IDS = [
    value.strip()
    for value in os.environ.get("OBJECTS_VIDEO_IDS", "").split(",")
    if value.strip()
]
VIDEO_OFFSET = int(os.environ.get("OBJECTS_VIDEO_OFFSET", "0"))
VIDEO_LIMIT = int(os.environ.get("OBJECTS_VIDEO_LIMIT", "0"))

print("model:", MODEL_NAME, "| conf:", CONF, "| iou:", IOU, flush=True)
print("imgsz:", IMGSZ, "| max_det:", MAX_DET, "| batch:", BATCH_SIZE, flush=True)
print("decode threads:", NUM_WORKERS, "| progress every:", PROGRESS_EVERY, "batches", flush=True)
print("classes filter:", CLASSES or "(all COCO)", flush=True)
print("bundle override:", BUNDLE_DIR, flush=True)
print("work:", WORK_DIR, flush=True)
print("shards:", SHARDS_DIR, flush=True)
print("zip:", ZIP_PATH, flush=True)
print(
    "video_ids:", VIDEO_IDS or "(all)",
    "| offset:", VIDEO_OFFSET,
    "| limit:", VIDEO_LIMIT,
    flush=True,
)


## 2. Cài dependency

Kaggle đã có `torch`; cell này bổ sung `ultralytics` (YOLO11). Model weights COCO
(`yolo11n/s/m/l/x.pt`) được tải tự động ở cell load model.


In [ ]:
import subprocess
import sys

packages = ["ultralytics>=8.3.0"]
if os.environ.get("OBJECTS_SKIP_INSTALL") != "1":
    print("installing:", " ".join(packages), flush=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        check=True,
    )
    print("install done", flush=True)
else:
    print("skip install (OBJECTS_SKIP_INSTALL=1)", flush=True)


## 3. Tìm bundle keyframe

Tìm `manifest.jsonl` có thư mục `keyframes/` đi kèm trong `/kaggle/input`.
Manifest chứa `kf_uid`, đường dẫn ảnh và metadata; không cần đọc SQLite.


In [ ]:
import json

candidates = []
seen = set()


def add_bundle(candidate: Path) -> None:
    bundle = candidate.resolve()
    if bundle in seen:
        return
    seen.add(bundle)
    if (bundle / "manifest.jsonl").is_file() and (bundle / "keyframes").is_dir():
        candidates.append(bundle)


if BUNDLE_DIR:
    add_bundle(Path(BUNDLE_DIR))
else:
    for root in (Path("/kaggle/input"), Path.cwd()):
        if not root.is_dir():
            continue
        for manifest in sorted(root.rglob("manifest.jsonl")):
            add_bundle(manifest.parent)

if not candidates:
    raise FileNotFoundError(
        "Không tìm thấy manifest.jsonl + keyframes/. "
        "Set OBJECTS_BUNDLE_DIR hoặc kiểm tra lại Kaggle dataset."
    )

BUNDLE_DIR = candidates[0]
MANIFEST_PATH = BUNDLE_DIR / "manifest.jsonl"

rows = []
for line in MANIFEST_PATH.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line:
        rows.append(json.loads(line))
if not rows:
    raise RuntimeError("manifest.jsonl rỗng")

print("bundle:", BUNDLE_DIR)
print("keyframes:", BUNDLE_DIR / "keyframes")
print("manifest rows:", f"{len(rows):,}")
if len(candidates) > 1:
    print("bỏ qua bundle khác:", [str(path) for path in candidates[1:]])


## 4. Chọn batch video

Thứ tự video lấy theo `video_id` đã sort. `OBJECTS_VIDEO_OFFSET`/`OBJECTS_VIDEO_LIMIT`
cho phép chạy lô 100 video mỗi lần mà không cần sửa notebook.


In [ ]:
by_video: dict[str, list[dict]] = {}
for row in sorted(rows, key=lambda item: (str(item["video_id"]), int(item["kf_uid"]))):
    by_video.setdefault(str(row["video_id"]), []).append(row)

all_video_ids = sorted(by_video)
if VIDEO_IDS:
    selected = list(dict.fromkeys(VIDEO_IDS))
    missing_videos = [video_id for video_id in selected if video_id not in by_video]
    if missing_videos:
        raise ValueError(f"OBJECTS_VIDEO_IDS không có trong manifest: {missing_videos}")
elif VIDEO_LIMIT > 0:
    selected = all_video_ids[VIDEO_OFFSET : VIDEO_OFFSET + VIDEO_LIMIT]
else:
    selected = all_video_ids[VIDEO_OFFSET:]

if not selected:
    raise RuntimeError("Không có video nào được chọn.")

total_selected = sum(len(by_video[video_id]) for video_id in selected)
print(f"videos in manifest: {len(all_video_ids):,}")
print(f"selected videos: {len(selected):,} | keyframes: {total_selected:,}")
print(f"batch: offset={VIDEO_OFFSET} limit={VIDEO_LIMIT} explicit={len(VIDEO_IDS)}")
if selected:
    print("first:", selected[0], "| last:", selected[-1])
if not VIDEO_IDS and VIDEO_LIMIT > 0:
    print("next OBJECTS_VIDEO_OFFSET:", VIDEO_OFFSET + len(selected))


## 5. Load YOLO11 COCO

**Accelerator:** nên bật GPU trong Kaggle Settings (T4 x2 hoặc P100) và Internet.
Notebook mặc định yêu cầu GPU; chạy CPU được bằng `OBJECTS_REQUIRE_GPU=0` nhưng rất chậm.
Trên CPU, precision (`quantize`/`half`) luôn bị tắt và không được truyền vào `model.predict`
(tránh lỗi `"half" not supported on CPU`). Notebook tự dùng `quantize=16` cho Ultralytics mới
và fallback `half=True` cho bản cũ.

`yolo11x.pt` là bản lớn nhất; đổi `OBJECTS_MODEL` sang `yolo11l.pt`/`yolo11m.pt` nếu
muốn nhanh hơn. `OBJECTS_CLASSES` (ví dụ `person,car,bus,truck`) giới hạn lớp cần detect.
Nếu GPU 16GB bị OOM, giảm `OBJECTS_BATCH_SIZE` (mặc định 16) hoặc `OBJECTS_IMGSZ`.


In [ ]:
import sys

print("python:", sys.version.split()[0], flush=True)

import torch

print("torch:", torch.__version__, flush=True)

from ultralytics import YOLO

print("ultralytics:", __import__("ultralytics").__version__, flush=True)

try:
    from ultralytics.cfg import DEFAULT_CFG_DICT

    HAS_QUANTIZE = "quantize" in DEFAULT_CFG_DICT
except ImportError:
    HAS_QUANTIZE = False
print("precision arg:", "quantize" if HAS_QUANTIZE else "half", flush=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
USE_HALF = DEVICE != "cpu"
if DEVICE == "cpu":
    print("device: cpu | fp16: False", flush=True)
    if REQUIRE_GPU:
        raise RuntimeError(
            "GPU không khả dụng — bật Accelerator=GPU trong Kaggle Settings "
            "(T4 x2 hoặc P100) và bật Internet. "
            "Đặt OBJECTS_REQUIRE_GPU=0 để chạy CPU (rất chậm, half sẽ tắt)."
        )
else:
    print(
        "device:", torch.cuda.get_device_name(0), "| fp16:", USE_HALF,
        flush=True,
    )

print("loading model:", MODEL_NAME, "(lần đầu sẽ tải weights)", flush=True)
model = YOLO(MODEL_NAME)
raw_names = model.names
NAMES = {
    int(index): str(name)
    for index, name in (
        raw_names.items() if isinstance(raw_names, dict) else enumerate(raw_names)
    )
}
print("classes:", len(NAMES), flush=True)

CLASS_IDS = None
if CLASSES:
    by_name = {name.lower(): index for index, name in NAMES.items()}
    unknown = [name for name in CLASSES if name not in by_name]
    if unknown:
        raise ValueError(
            f"OBJECTS_CLASSES có nhãn không thuộc COCO: {unknown}. "
            f"Ví dụ hợp lệ: {sorted(by_name)[:20]}"
        )
    CLASS_IDS = [by_name[name] for name in CLASSES]
    print("class filter:", CLASSES, flush=True)


## 6. Detect từng video thành shard

Mỗi video ghi vào `SHARDS_DIR/<video_id>/objects.jsonl`. Shard đã xong được bỏ qua khi
resume; shard dở dang được cắt về đúng số dòng hợp lệ rồi chạy tiếp.


In [ ]:
import hashlib
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import UTC, datetime

from PIL import Image


def load_keyframe(row: dict) -> Image.Image:
    with Image.open(BUNDLE_DIR / row["path"]) as image:
        return image.convert("RGB")


def write_json(path: Path, payload: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)


def make_record(row: dict, counts: dict[str, int]) -> dict:
    labels = sorted(counts)
    return {
        "kf_uid": int(row["kf_uid"]),
        "video_id": str(row["video_id"]),
        "generation": int(row.get("generation", 0)),
        "shot_idx": int(row["shot_idx"]),
        "kf_idx": int(row["kf_idx"]),
        "ts_ms": int(row["ts_ms"]),
        "objects": labels,
        "counts": {label: int(counts[label]) for label in labels},
    }


def detect_batch(images: list[Image.Image]) -> list[dict[str, int]]:
    predict_kwargs = {
        "conf": CONF,
        "iou": IOU,
        "imgsz": IMGSZ,
        "max_det": MAX_DET,
        "device": DEVICE,
        "verbose": False,
    }
    if USE_HALF:
        if HAS_QUANTIZE:
            predict_kwargs["quantize"] = 16
        else:
            predict_kwargs["half"] = True
    if CLASS_IDS is not None:
        predict_kwargs["classes"] = CLASS_IDS
    try:
        results = model.predict(images, **predict_kwargs)
    except (RuntimeError, NotImplementedError) as exc:
        message = str(exc).lower()
        precision_key = next(
            (name for name in ("quantize", "half") if name in predict_kwargs),
            None,
        )
        if precision_key is None or not any(
            word in message for word in ("half", "quantize", "fp16")
        ):
            raise
        predict_kwargs.pop(precision_key)
        print(
            f"{precision_key} không hỗ trợ trên thiết bị này, chạy lại với fp32",
            flush=True,
        )
        results = model.predict(images, **predict_kwargs)
    batch_counts = []
    for result in results:
        counts: dict[str, int] = {}
        boxes = result.boxes
        if boxes is not None and len(boxes) > 0:
            for cls_id in boxes.cls.detach().cpu().numpy().astype(int):
                label = NAMES.get(int(cls_id))
                if label:
                    counts[label] = counts.get(label, 0) + 1
        batch_counts.append(counts)
    return batch_counts


def shard_manifest(
    video_id: str,
    total: int,
    plan_hash: str,
    counts_by_label: dict[str, int],
) -> dict:
    return {
        "version": 1,
        "video_id": video_id,
        "created_at": datetime.now(UTC).isoformat(),
        "model": MODEL_NAME,
        "conf": CONF,
        "iou": IOU,
        "imgsz": IMGSZ,
        "max_det": MAX_DET,
        "classes": CLASSES or None,
        "count": total,
        "plan_hash": plan_hash,
        "by_label": dict(sorted(counts_by_label.items())),
        "files": {"objects": "objects.jsonl"},
    }


decode_pool = ThreadPoolExecutor(max_workers=NUM_WORKERS)
shard_summaries = []
overall_started = time.time()
print(f"detecting {len(selected):,} videos -> {SHARDS_DIR}", flush=True)

for position, video_id in enumerate(selected, start=1):
    video_rows = by_video[video_id]
    total = len(video_rows)
    uids = [int(row["kf_uid"]) for row in video_rows]
    plan_hash = hashlib.sha256(
        (
            f"{MODEL_NAME}|{CONF}|{IOU}|{IMGSZ}|{MAX_DET}|"
            f"{','.join(CLASSES)}|{','.join(map(str, uids))}"
        ).encode()
    ).hexdigest()

    shard_dir = SHARDS_DIR / video_id
    shard_dir.mkdir(parents=True, exist_ok=True)
    objects_path = shard_dir / "objects.jsonl"
    manifest_path = shard_dir / "manifest.json"
    plan_path = shard_dir / "plan.json"
    progress_path = shard_dir / "progress.json"

    rows_done = 0
    counts_by_label: dict[str, int] = {}
    reuse = False

    if RESUME and objects_path.is_file() and plan_path.is_file():
        try:
            previous = json.loads(plan_path.read_text(encoding="utf-8"))
            plan_ok = (
                previous.get("plan_hash") == plan_hash
                and int(previous.get("count", -1)) == total
            )
        except (OSError, ValueError):
            plan_ok = False

        if plan_ok:
            lines = objects_path.read_text(encoding="utf-8").splitlines()
            valid = []
            for index, line in enumerate(lines):
                if index >= total:
                    break
                try:
                    record = json.loads(line)
                except ValueError:
                    break
                if int(record.get("kf_uid", -1)) != uids[index]:
                    break
                valid.append(line)
                for label, value in (record.get("counts") or {}).items():
                    counts_by_label[label] = counts_by_label.get(label, 0) + int(value)
            rows_done = len(valid)
            reuse = True
            if rows_done != len(lines):
                tmp = objects_path.with_suffix(".jsonl.tmp")
                tmp.write_text(
                    "\n".join(valid) + ("\n" if valid else ""), encoding="utf-8"
                )
                os.replace(tmp, objects_path)

    if not reuse:
        objects_path.write_text("", encoding="utf-8")
        manifest_path.unlink(missing_ok=True)
        plan_path.unlink(missing_ok=True)
        rows_done = 0
        counts_by_label = {}

    write_json(
        plan_path,
        {
            "video_id": video_id,
            "model": MODEL_NAME,
            "conf": CONF,
            "iou": IOU,
            "imgsz": IMGSZ,
            "max_det": MAX_DET,
            "classes": CLASSES or None,
            "count": total,
            "plan_hash": plan_hash,
        },
    )

    if reuse and rows_done == total:
        write_json(
            manifest_path,
            shard_manifest(video_id, total, plan_hash, counts_by_label),
        )
        print(f"[{position}/{len(selected)}] {video_id}: skip (done)")
        shard_summaries.append({"video_id": video_id, "count": total, "skipped": True})
        continue

    write_json(progress_path, {"rows_done": rows_done, "status": "running"})
    started = time.time()
    print(
        f"[{position}/{len(selected)}] {video_id}: start "
        f"({rows_done:,}/{total:,} keyframes)",
        flush=True,
    )

    def load_batch(start: int) -> list[Image.Image]:
        return [load_keyframe(row) for row in video_rows[start : start + BATCH_SIZE]]

    pending = decode_pool.submit(load_batch, rows_done) if rows_done < total else None
    with objects_path.open("a", encoding="utf-8") as handle:
        for batch_index, start in enumerate(range(rows_done, total, BATCH_SIZE), start=1):
            images = pending.result()
            upcoming = start + BATCH_SIZE
            pending = decode_pool.submit(load_batch, upcoming) if upcoming < total else None

            batch_counts = detect_batch(images)
            for offset, counts in enumerate(batch_counts):
                record = make_record(video_rows[start + offset], counts)
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                for label, value in counts.items():
                    counts_by_label[label] = counts_by_label.get(label, 0) + value
            handle.flush()

            if batch_index % PROGRESS_EVERY == 0:
                write_json(
                    progress_path,
                    {
                        "rows_done": min(start + len(batch_counts), total),
                        "status": "running",
                    },
                )

    write_json(progress_path, {"rows_done": total, "status": "done"})
    write_json(
        manifest_path,
        shard_manifest(video_id, total, plan_hash, counts_by_label),
    )
    shard_summaries.append({"video_id": video_id, "count": total, "skipped": False})
    print(
        f"[{position}/{len(selected)}] {video_id}: {total:,} keyframes "
        f"in {time.time() - started:,.1f}s",
        flush=True,
    )

print(
    f"done {len(shard_summaries):,} videos in "
    f"{(time.time() - overall_started) / 60:,.1f} min",
    flush=True,
)


## 7. Đóng gói shard

Zip chỉ chứa `shards/<video_id>/objects.jsonl` + `manifest.json` của từng video
(không chứa keyframes, không chứa `progress.json`).


In [ ]:
import zipfile

label_totals: dict[str, int] = {}
total_keyframes = 0
for summary in shard_summaries:
    shard_dir = SHARDS_DIR / summary["video_id"]
    manifest = json.loads((shard_dir / "manifest.json").read_text(encoding="utf-8"))
    total_keyframes += int(manifest["count"])
    for label, value in manifest.get("by_label", {}).items():
        label_totals[label] = label_totals.get(label, 0) + int(value)

run_manifest = {
    "version": 1,
    "created_at": datetime.now(UTC).isoformat(),
    "model": MODEL_NAME,
    "conf": CONF,
    "iou": IOU,
    "imgsz": IMGSZ,
    "max_det": MAX_DET,
    "classes": CLASSES or None,
    "videos": len(shard_summaries),
    "keyframes": total_keyframes,
    "by_label": dict(sorted(label_totals.items())),
}

README = f"""AIC2024 KIS - YOLO11 COCO object shards
model: {MODEL_NAME}
conf: {CONF} | iou: {IOU} | imgsz: {IMGSZ}

Import locally with:
  uv run ir import-objects <zip>
This extracts to state/objects/shards/<video_id>/ and fills the SQLite
keyframe_objects table used by the search filter.
"""

ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
written = 0
with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
) as archive:
    for summary in shard_summaries:
        shard_dir = SHARDS_DIR / summary["video_id"]
        for path in sorted(shard_dir.glob("*")):
            if not path.is_file() or path.name in {"progress.json", "plan.json"}:
                continue
            archive.write(
                path,
                arcname=f"shards/{summary['video_id']}/{path.name}",
            )
            written += 1
    archive.writestr("manifest.json", json.dumps(run_manifest, indent=2, ensure_ascii=False))
    archive.writestr("README.txt", README)

print(f"zip: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:,.1f} MB, {written} files)")


## 8. Kiểm tra batch

Verify số dòng `objects.jsonl` khớp `manifest.count` và in top nhãn xuất hiện nhiều nhất.


In [ ]:
total_keyframes = 0
label_totals = {}
for summary in shard_summaries:
    shard_dir = SHARDS_DIR / summary["video_id"]
    manifest = json.loads((shard_dir / "manifest.json").read_text(encoding="utf-8"))
    with (shard_dir / "objects.jsonl").open(encoding="utf-8") as handle:
        count = sum(1 for line in handle if line.strip())
    assert count == int(manifest["count"]), (
        summary["video_id"],
        count,
        manifest["count"],
    )
    total_keyframes += count
    for label, value in manifest.get("by_label", {}).items():
        label_totals[label] = label_totals.get(label, 0) + int(value)
    print(f"{summary['video_id']}: {count:,} keyframes")

print(f"batch: {len(shard_summaries):,} videos, {total_keyframes:,} keyframes")
print("top labels:")
for label, value in sorted(label_totals.items(), key=lambda item: -item[1])[:30]:
    print(f"  {label:>20}: {value:,}")
if not VIDEO_IDS and VIDEO_LIMIT > 0:
    print("next OBJECTS_VIDEO_OFFSET:", VIDEO_OFFSET + len(selected))


## Sau khi chạy

1. Tải `artifacts_objects_shards.zip`.
2. Ở local chạy `uv run ir import-objects artifacts_objects_shards.zip`.
   Lệnh này giải nén vào `state/objects/shards/<video_id>/` và ghi nhãn + count vào
   SQLite (`keyframes.objects`/`obj_count` và bảng `keyframe_objects`).
3. Kiểm tra bằng `uv run ir objects`.
4. Filter khi search: `uv run ir search "..." --object person` hoặc chọn chip object
   trên web UI. Nhiều nhãn là AND (keyframe phải có đủ các nhãn đó).
5. Import lại cùng zip an toàn (idempotent): nhãn của keyframe được thay thế, không bị cộng dồn.
